# Module 00 · Course setup

This notebook checks that your Mac (or Colab) is ready for the course. Run the cells **in order, top to bottom**. Each has a note above (*what this does*) and below (*what you just saw*). It uses only Python's standard library, so there is nothing to install yet.

| Section | What it checks |
|---|---|
| Setup | Finds the course folder (and downloads it in Colab) |
| 1 | How the course works *(reading only)* |
| 2 | Tools |
| 3 | Your two API keys |
| 4 | The sample documents |
| Checkpoint | Three questions |

## Setup
**What this cell does:** in Colab it downloads the course. On a Mac it simply finds the repo folder, two levels up from this notebook.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = "https://github.com/panwars87/sha-genai-learning.git"
    BRANCH = "main"   # change if you are studying a different branch
    if not os.path.isdir("/content/sha-genai-learning"):
        !git clone -q -b {BRANCH} {REPO} /content/sha-genai-learning
    %cd /content/sha-genai-learning/modules/00-course-setup

REPO_ROOT = Path.cwd().parents[1]
print("Course folder:", REPO_ROOT)

**What you just saw:** the path to the course folder. It should end in `sha-genai-learning`.

---
## Section 1 · How the course works *(reading only)*

Every module follows the same four steps:

1. **Read** the module's HTML chapter, one section at a time.
2. **Run** the module notebook. Its sections match the chapter's sections.
3. **Break it.** Each notebook ends with experiments that fail on purpose.
4. **Checkpoint.** Answer the questions before moving on.

Each module folder holds:

| File | What it is |
|---|---|
| `chapter.html` | The chapter (also published online) |
| `NN-name.ipynb` | The notebook you run alongside the chapter |
| `app/` | DocuMind as it stands at the end of this module |
| `NOTES.md` | Your notes |

---
## Section 2 · Tools
**What this cell does:** checks your Python version, and that `uv` and `git` are installed.

In [ ]:
import shutil

ok = sys.version_info >= (3, 12)
print(f"{'✓' if ok else '✗'} Python {sys.version.split()[0]} (need 3.12 or newer)")
for tool in ("uv", "git"):
    found = shutil.which(tool)
    if IN_COLAB and tool == "uv":
        print("· uv: not needed in Colab")
        continue
    print(f"{'✓' if found else '✗'} {tool}: {found or 'not found. See the chapter, section 2.'}")

**What you just saw:** three ticks on a Mac. Install anything marked ✗ with the chapter's commands (`brew install uv`, `xcode-select --install`), then run the cell again.

---
## Section 3 · Your two API keys
**What this cell does:** checks that `ANTHROPIC_API_KEY` and `OPENAI_API_KEY` each have a value. On a Mac it looks in the repo's `.env` file; in Colab it looks in Colab Secrets (the key icon in the left sidebar). **It only checks that a value exists. It never prints one.**

In [ ]:
KEY_NAMES = ("ANTHROPIC_API_KEY", "OPENAI_API_KEY")

def env_file_names(path):
    """Names in a .env file that have a non-empty value."""
    names = set()
    for line in path.read_text().splitlines():
        name, _, value = line.partition("=")
        if not line.strip().startswith("#") and value.split("#")[0].strip():
            names.add(name.strip())
    return names

if IN_COLAB:
    from google.colab import userdata
    present = set()
    for name in KEY_NAMES:
        try:
            if userdata.get(name):
                present.add(name)
        except Exception:
            pass
else:
    env_file = REPO_ROOT / ".env"
    if not env_file.exists():
        print("✗ No .env file yet. In a terminal, from the course folder: cp .env.example .env")
    present = env_file_names(env_file) if env_file.exists() else set()

for name in KEY_NAMES:
    print(f"{'✓' if name in present else '✗'} {name}")

**What you just saw:** two ticks means you're ready. Keys come from console.anthropic.com and platform.openai.com. Set a monthly spending limit in both consoles; the course also has its own cap.

---
## Section 4 · The sample documents
**What this cell does:** previews the made-up invoices and receipts that DocuMind will learn on.

In [ ]:
for folder in ("invoices", "receipts"):
    for path in sorted((REPO_ROOT / "data" / folder).glob("*.txt")):
        text = path.read_text()
        print(f"──── {folder}/{path.name}  ({len(text):,} characters ≈ {len(text) // 4} tokens)")
        print("\n".join(text.strip().splitlines()[:6]), "\n   …\n")

**What you just saw:** four very different layouts: two currencies, tax and VAT, a tip, a discount, cash and change. A fixed template can't read all of them; a model can. That variety is why DocuMind uses an LLM. Every total adds up, so later modules can check the maths.

---
## Checkpoint

<details><summary>1. Why is <code>.env</code> in <code>.gitignore</code>?</summary>
So keys never reach git. A key that has been pushed counts as leaked: revoke it and create a new one.
</details>

<details><summary>2. Which code reads your API keys?</summary>
Only the LiteLLM library inside DocuMind's gateway module. Feature code never touches them.
</details>

<details><summary>3. Why use made-up documents?</summary>
Real invoices contain personal and commercial data. Learning never needs it, and in a bank it would break data-handling rules.
</details>

All green? Open **Module 01**.